# Case Study 3: Text Summarization

**PAB-MMR**: Polarity- and Aspect-Balanced MMR. Evaluated against human references on FewSum and on telecom digests where no references exist.

In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

PACKAGES = ["rouge-score", "sentencepiece", "protobuf"]
KAGGLE = os.path.isdir("/kaggle/working")
if KAGGLE:
    if PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
    hits = glob.glob("/kaggle/input/**/telelens/__init__.py", recursive=True)
    if hits:
        ROOT = Path(hits[0]).parents[2]
    else:
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/BharatChandra-sys/NLP-case-study.git", "/tmp/repo"], check=True)
        ROOT = Path("/tmp/repo")
    RESULTS = Path("/kaggle/working/results/summarization")
else:
    ROOT = Path.cwd().resolve().parent
    RESULTS = ROOT / "results" / "summarization"
sys.path.insert(0, str(ROOT / "src"))
RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


def save_json(name, payload):
    (RESULTS / name).write_text(json.dumps(payload, indent=2, default=float), encoding="utf-8")


In [ ]:
import itertools
from scipy.stats import wilcoxon
from telelens import summarize as S
from telelens.data import load_fewsum, load_telecom
from telelens.embeddings import log_odds_polarity, polarity_counterfit, train_sgns
from telelens.metrics import bootstrap_ci, js_binary
from telelens.punctuate import PunctuationRestorer
from telelens.sentiment import grouped_folds, make_vectorizer, pretrained_sst2
from telelens.text import bind_negation, boundary_prf, segment_fixed, segment_unpunctuated, split_sentences, tokenize
from telelens import viz
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
import re

dev_groups = load_fewsum("train") + load_fewsum("val")
test_groups = load_fewsum("test")
BUDGET = int(np.mean([S.word_count(r) for g in dev_groups for r in g["references"]]))
print(len(dev_groups), len(test_groups), "budget", BUDGET)

## FewSum: reviews, human references and a dev/test split

In [ ]:
all_sentences = {}
def prepare(group):
    sentences = S.filter_sentences([s for r in group["reviews"] for s in split_sentences(r)])
    vectors = S.sentence_vectors(sentences)
    return {"sentences": sentences, "vectors": vectors, "aspects": S.kmeans_aspects(vectors), "references": group["references"],
            "reviews": group["reviews"], "target": float(np.mean([(r - 1) / 4 for r in group["ratings"]]))}
prepared_dev = [prepare(g) for g in dev_groups]
prepared_test = [prepare(g) for g in test_groups]
flat = [s for g in prepared_dev + prepared_test for s in g["sentences"]]
scores = pretrained_sst2(flat, DEVICE)
cursor = 0
for g in prepared_dev + prepared_test:
    g["polarity"] = scores[cursor : cursor + len(g["sentences"])]
    cursor += len(g["sentences"])


def run_pab(g, lam, beta, eta):
    return S.pab_mmr(g["vectors"], g["sentences"], BUDGET, polarity=g["polarity"], aspects=g["aspects"], target=g["target"], lam=lam, beta=beta, eta=eta)


def text_of(g, chosen):
    return " ".join(g["sentences"][i] for i in chosen)


grid = []
for lam, beta, eta in itertools.product([0.3, 0.5, 0.7, 0.9], [0.0, 0.1, 0.3], [0.0, 0.25, 0.5, 1.0]):
    r2 = np.mean([S.rouge_multi(text_of(g, run_pab(g, lam, beta, eta)), g["references"])["rouge2"] for g in prepared_dev])
    grid.append({"lam": lam, "beta": beta, "eta": eta, "dev_rouge2": r2})
grid = pd.DataFrame(grid).sort_values("dev_rouge2", ascending=False)
grid.to_csv(RESULTS / "fewsum_dev_grid.csv", index=False)
best_pab = grid.iloc[0]
best_mmr = grid[(grid.beta == 0) & (grid.eta == 0)].iloc[0]
FIXED = dict(lam=float(best_mmr.lam), beta=0.3, eta=0.5)
print(best_pab.to_dict(), best_mmr.to_dict(), FIXED)
grid.head(6).round(4)

In [ ]:
documents = [" ".join(g["reviews"]) for g in prepared_test]
bart = S.bart_summaries(documents, DEVICE)
def fewsum_outputs(g, i):
    v, s = g["vectors"], g["sentences"]
    return {"Lead": text_of(g, S.lead(s, BUDGET)), "Centroid": text_of(g, S.centroid(v, s, BUDGET)), "TextRank": text_of(g, S.textrank(v, s, BUDGET)),
            "MMR (tuned)": text_of(g, run_pab(g, best_mmr.lam, 0.0, 0.0)), "PAB-MMR (tuned)": text_of(g, run_pab(g, best_pab.lam, best_pab.beta, best_pab.eta)),
            "PAB-MMR (fixed)": text_of(g, run_pab(g, **FIXED)), "BART-large-CNN": bart[i]}
rows, gaps, samples = [], [], []
for i, g in enumerate(prepared_test):
    outputs = fewsum_outputs(g, i)
    for name, summary in outputs.items():
        rows.append({"group": i, "method": name, "words": S.word_count(summary), **S.rouge_multi(summary, g["references"])})
    samples.append({"group": test_groups[i]["group"], **outputs, "reference": g["references"][0]})
fewsum = pd.DataFrame(rows)
fewsum.to_csv(RESULTS / "fewsum_by_group.csv", index=False)
agg = fewsum.drop(columns="group").groupby("method", sort=False).mean()
agg["rouge2_lo"], agg["rouge2_hi"] = zip(*[bootstrap_ci(fewsum[fewsum.method == m].rouge2) for m in agg.index])
agg.to_csv(RESULTS / "fewsum_scores.csv")
tests = []
for a, b in [("PAB-MMR (tuned)", "MMR (tuned)"), ("PAB-MMR (fixed)", "MMR (tuned)"), ("PAB-MMR (tuned)", "TextRank"), ("PAB-MMR (tuned)", "BART-large-CNN")]:
    x, y = fewsum[fewsum.method == a].rouge2.to_numpy(), fewsum[fewsum.method == b].rouge2.to_numpy()
    tests.append({"a": a, "b": b, "mean_diff": (x - y).mean(), "wilcoxon_p": wilcoxon(x, y).pvalue if np.any(x != y) else 1.0})
pd.DataFrame(tests).to_csv(RESULTS / "fewsum_tests.csv", index=False)
pd.DataFrame(samples).to_csv(RESULTS / "fewsum_samples.csv", index=False)
display(agg.round(4), pd.DataFrame(tests).round(4))

## Segmenting text that has no punctuation: measured by stripping FewSum punctuation

In [ ]:
strip = lambda s: " ".join(re.sub(r"[^a-z\s]", " ", s.lower()).split())
restorer = PunctuationRestorer(DEVICE)
truths, joined = [], []
for g in prepared_dev + prepared_test:
    for review in g["reviews"]:
        truth = [t for t in (strip(x) for x in split_sentences(review)) if t]
        if len(truth) > 1:
            truths.append(truth)
            joined.append(" ".join(truth))
restored = restorer.segment_many(joined)
candidates = {"Cue words": [segment_unpunctuated(t) for t in joined], "Fixed 15-word windows": [segment_fixed(t) for t in joined], "Punctuation restoration": restored}
segmentation = pd.DataFrame([{"method": m, **dict(zip(["precision", "recall", "f1"], np.mean([boundary_prf(t, p) for t, p in zip(truths, parts)], axis=0))), "reviews": len(truths)} for m, parts in candidates.items()])
stripped = {}
for method in candidates:
    values = []
    for g in prepared_test:
        cleaned = [strip(r) for r in g["reviews"]]
        if method == "Cue words":
            pieces = [segment_unpunctuated(c) for c in cleaned]
        elif method == "Fixed 15-word windows":
            pieces = [segment_fixed(c) for c in cleaned]
        else:
            pieces = restorer.segment_many(cleaned)
        sentences = S.filter_sentences([x for part in pieces for x in part])
        chosen = S.pab_mmr(S.sentence_vectors(sentences), sentences, BUDGET, lam=float(best_mmr.lam))
        values.append(S.rouge_multi(" ".join(sentences[i] for i in chosen), [strip(r) for r in g["references"]])["rouge2"])
    stripped[method] = float(np.mean(values))
segmentation.to_csv(RESULTS / "segmentation.csv", index=False)
save_json("stripped_rouge2.json", stripped)
display(segmentation.round(4), stripped)

## Telecom digests: provider and provider-by-aspect clusters, with no references available

In [ ]:
frame = load_telecom(10000)
frame = frame.reset_index(drop=True)
plain = [tokenize(t) for t in frame["text"]]
bound = [bind_negation(d) for d in plain]
base, _ = train_sgns(bound, device=DEVICE, bound=True)
pa, _ = polarity_counterfit(base, log_odds_polarity(bound, frame["label"]), device=DEVICE)
lexicon = S.expand_aspects(pa)
save_json("aspect_lexicon.json", {k: sorted(v) for k, v in lexicon.items()})
review_vectorizers = []
segment_rows = []
for doc, pieces in enumerate(restorer.segment_many(frame["text"].tolist())):
    for seg in S.filter_sentences(pieces, min_words=5):
        segment_rows.append((doc, seg))
segments = pd.DataFrame(segment_rows, columns=["doc", "text"])
segments["polarity"] = 0.0
for tr, te in grouped_folds(frame):
    vectorizer = make_vectorizer()
    model = CalibratedClassifierCV(LogisticRegression(C=4.0, max_iter=2000), method="sigmoid", cv=3).fit(vectorizer.fit_transform(frame["text"].iloc[tr]), frame["label"].iloc[tr])
    mask = segments["doc"].isin(set(te))
    segments.loc[mask, "polarity"] = model.predict_proba(vectorizer.transform(segments.loc[mask, "text"]))[:, 1]
segments["aspect"] = S.lexicon_aspects(segments["text"].tolist(), lexicon)
global_vectorizer = make_vectorizer()
global_model = CalibratedClassifierCV(LogisticRegression(C=4.0, max_iter=2000), method="sigmoid", cv=3).fit(global_vectorizer.fit_transform(frame["text"]), frame["label"])
PROVIDERS = ["virgin", "bt", "talktalk", "vodafone", "ee", "sky", "tesco", "zen"]
rng = np.random.default_rng(SEED)
clusters = []
for provider in PROVIDERS:
    docs = frame.index[[provider in t.split() for t in frame["text"]]].to_numpy()
    clusters.append((provider, "all", docs))
    for aspect in S.ASPECT_SEEDS:
        chosen = segments[(segments.doc.isin(docs)) & (segments.aspect == aspect)].doc.unique()
        if len(chosen) >= 30:
            clusters.append((provider, aspect, chosen))
print(len(clusters), "clusters")

In [ ]:
rows, examples = [], []
bart_inputs, cluster_payload = [], []
for provider, aspect, docs in clusters:
    docs = rng.choice(docs, size=min(60, len(docs)), replace=False)
    part = segments[segments.doc.isin(set(docs))]
    if aspect != "all":
        part = part[part.aspect == aspect]
    sentences = part["text"].tolist()
    vectors = S.sentence_vectors(sentences)
    target = float(frame.loc[docs, "label"].mean())
    cluster_payload.append((provider, aspect, docs, sentences, vectors, part["polarity"].to_numpy(), part["aspect"].to_numpy(), S.kmeans_aspects(vectors, 4), target))
    bart_inputs.append(" ".join(frame.loc[docs, "text"].tolist())[:6000])
bart_telecom = S.bart_summaries(bart_inputs, DEVICE, max_length=70, min_length=30)
for c, (provider, aspect, docs, sentences, vectors, polarity, aspect_labels, sub, target) in enumerate(cluster_payload):
    facets = aspect_labels if aspect == "all" else sub
    centre = vectors.mean(0)
    picks = {"Lead": S.lead(sentences, BUDGET), "Centroid": S.centroid(vectors, sentences, BUDGET), "TextRank": S.textrank(vectors, sentences, BUDGET),
             "MMR": S.pab_mmr(vectors, sentences, BUDGET, lam=float(best_mmr.lam)),
             "PAB-MMR": S.pab_mmr(vectors, sentences, BUDGET, polarity=polarity, aspects=facets, target=target, **FIXED)}
    for name, chosen in picks.items():
        rows.append({"cluster": c, "provider": provider, "aspect": aspect, "method": name, "target_share_good": target,
                     "polarity_gap": S.polarity_gap(polarity, sentences, chosen, target), "aspect_recall": S.aspect_recall(facets, chosen),
                     "redundancy": S.redundancy(vectors, chosen), "words": sum(S.word_count(sentences[i]) for i in chosen),
                     "centroid_cosine": float(np.dot(vectors[chosen].mean(0), centre) / (np.linalg.norm(vectors[chosen].mean(0)) * np.linalg.norm(centre) + 1e-9))})
    summary = bart_telecom[c]
    bart_segments = S.filter_sentences(split_sentences(summary)) or [summary]
    bart_polarity = global_model.predict_proba(global_vectorizer.transform(bart_segments))[:, 1]
    rows.append({"cluster": c, "provider": provider, "aspect": aspect, "method": "BART-large-CNN", "target_share_good": target, "polarity_gap": js_binary(float(bart_polarity.mean()), target),
                 "aspect_recall": np.nan, "redundancy": np.nan, "words": S.word_count(summary), "centroid_cosine": np.nan, "novel_token_rate": S.novel_token_rate(summary, bart_inputs[c])})
    if aspect == "all":
        examples.append({"provider": provider, "share_good": target, "PAB-MMR": " | ".join(sentences[i] for i in picks["PAB-MMR"]), "MMR": " | ".join(sentences[i] for i in picks["MMR"]), "BART": summary})
telecom = pd.DataFrame(rows)
telecom.to_csv(RESULTS / "telecom_by_cluster.csv", index=False)
telecom_agg = telecom.groupby("method", sort=False)[["polarity_gap", "aspect_recall", "redundancy", "centroid_cosine", "words", "novel_token_rate"]].mean()
telecom_agg.to_csv(RESULTS / "telecom_scores.csv")
pd.DataFrame(examples).to_csv(RESULTS / "telecom_digests.csv", index=False)
tests = []
for metric, base_name in [("polarity_gap", "MMR"), ("aspect_recall", "MMR"), ("redundancy", "MMR"), ("polarity_gap", "TextRank"), ("polarity_gap", "BART-large-CNN")]:
    a = telecom[telecom.method == "PAB-MMR"].set_index("cluster")[metric]
    b = telecom[telecom.method == base_name].set_index("cluster")[metric]
    tests.append({"metric": metric, "versus": base_name, "pab_mean": a.mean(), "other_mean": b.mean(), "wilcoxon_p": wilcoxon(a, b).pvalue, "clusters": len(a)})
pd.DataFrame(tests).to_csv(RESULTS / "telecom_tests.csv", index=False)
gap = telecom_agg["polarity_gap"]
viz.hbar(RESULTS / "telecom_polarity_gap.png", gap.index.tolist(), gap.tolist(), "Lower is better: sentiment drift from the review cluster", "Jensen-Shannon divergence", highlight="PAB-MMR")
r2 = agg["rouge2"].sort_values(ascending=False)
viz.hbar(RESULTS / "fewsum_rouge2.png", r2.index.tolist(), r2.tolist(), "ROUGE-2 F1 against human references, FewSum test", "ROUGE-2 F1", highlight="PAB-MMR (tuned)")
display(telecom_agg.round(4), pd.DataFrame(tests).round(4), pd.DataFrame(examples).head(3))

## Recorded Kaggle results

Executed on a Kaggle T4 GPU as `telelens-03-summarization`. Files are in `results/summarization/`.

**fewsum_scores.csv**

| method | words | rouge1 | rouge2 | rougeL | rouge2_lo | rouge2_hi |
|---|---|---|---|---|---|---|
| Lead | 48.7 | 0.2668 | 0.0414 | 0.1554 | 0.0294 | 0.055 |
| Centroid | 48.55 | 0.2776 | 0.0475 | 0.1646 | 0.0365 | 0.0586 |
| TextRank | 48.3 | 0.2806 | 0.0454 | 0.1637 | 0.0359 | 0.0552 |
| MMR (tuned) | 48.7 | 0.2902 | 0.0441 | 0.1672 | 0.0332 | 0.0553 |
| PAB-MMR (tuned) | 48.5 | 0.277 | 0.0451 | 0.1634 | 0.0346 | 0.0556 |
| PAB-MMR (fixed) | 49.25 | 0.2878 | 0.047 | 0.1632 | 0.0359 | 0.0584 |
| BART-large-CNN | 41.9 | 0.2654 | 0.0435 | 0.1633 | 0.0292 | 0.0608 |

**fewsum_tests.csv**

| a | b | mean_diff | wilcoxon_p |
|---|---|---|---|
| PAB-MMR (tuned) | MMR (tuned) | 0.001 | 0.408 |
| PAB-MMR (fixed) | MMR (tuned) | 0.0029 | 0.5228 |
| PAB-MMR (tuned) | TextRank | -0.0003 | 0.4446 |
| PAB-MMR (tuned) | BART-large-CNN | 0.0016 | 0.7285 |

**segmentation.csv**

| method | precision | recall | f1 | reviews |
|---|---|---|---|---|
| Cue words | 0.2537 | 0.1791 | 0.1942 | 453 |
| Fixed 15-word windows | 0.3199 | 0.3057 | 0.2994 | 453 |
| Punctuation restoration | 0.8178 | 0.887 | 0.8281 | 453 |

**telecom_scores.csv**

| method | polarity_gap | aspect_recall | redundancy | centroid_cosine | words | novel_token_rate |
|---|---|---|---|---|---|---|
| Lead | 0.0459 | 0.6528 | 0.03 | 0.2796 | 48.1064 |  |
| Centroid | 0.0738 | 0.5211 | 0.3166 | 0.5174 | 48.4043 |  |
| TextRank | 0.0648 | 0.5625 | 0.2904 | 0.5179 | 48.5106 |  |
| MMR | 0.0586 | 0.6795 | 0.0808 | 0.576 | 48.2979 |  |
| PAB-MMR | 0.0192 | 0.8504 | 0.0719 | 0.5447 | 48.617 |  |
| BART-large-CNN | 0.0805 |  |  |  | 60.1064 | 0.0236 |

**telecom_tests.csv**

| metric | versus | pab_mean | other_mean | wilcoxon_p | clusters |
|---|---|---|---|---|---|
| polarity_gap | MMR | 0.0192 | 0.0586 | 0.0 | 47 |
| aspect_recall | MMR | 0.8504 | 0.6795 | 0.0 | 47 |
| redundancy | MMR | 0.0719 | 0.0808 | 0.1334 | 47 |
| polarity_gap | TextRank | 0.0192 | 0.0648 | 0.0 | 47 |
| polarity_gap | BART-large-CNN | 0.0192 | 0.0805 | 0.0002 | 47 |

![fewsum_rouge2.png](../results/summarization/fewsum_rouge2.png)

![telecom_polarity_gap.png](../results/summarization/telecom_polarity_gap.png)

